In [1]:
from platform import python_version
print(python_version())

3.11.14


### Cluster with Tahoe or sc-GTP

### Huggingface: tahoebio/Tahoe-x1-embeddings

https://github.com/tahoebio/tahoe-x1

Tahoe-x1: Scaling Perturbation-Trained Single-Cell Foundation Models to 3 Billion Parameters


#### Memory

That's not a general "64 GB isn't enough" — swap is fully exhausted at 2.0G, which means something asked for tens of GB in one allocation. Given where you are in the pipeline, the culprit is almost certainly load_tahoe_de, and the arithmetic says so:

The DE table is ~4.09e9 rows over ~75k conditions × ~54k genes. 

Filtering to pancreas doesn't help much — roughly 
- 6 lines × 379 drugs × ~4 doses × 54k genes ≈ 5e8 rows, 
- materialised in pandas with gene/drug/cell_line_id as object-dtype strings (~200 B/row) before pivot_table ever runs. 
- That's >100 GB. full_Z and consensus_cluster are megabytes by comparison.



In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, ROOT_SRC)


if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import create_dir
from libs.MTD_lib import MTD
from libs.cBioPortal_lib import cBioPortal
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config
from libs.prism_lib import PRISM
from libs.prism_program_lib import *


from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(
/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'BRCA'
PSI_ID = 'ACC'
PSI_ID = 'CESC'
PSI_ID = 'PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/PAAD/config/all_lfc_cutoffs_PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", mtd.disease, case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/PAAD
>>> PAAD Tumor
>>> case Tumor
>>> psi_id or disease: PAAD
Error: No data available for the specified PAAD.
Error: could not find /home/flavio/uv/perturb_agent/data/TCGA/PAAD/lfc/PAAD_final_LFC_Tumor_x_CTRL_not_normalized.tsv
No dflfc table was calculated for this case Tumor

Echo Parameters:
	0/0 DEGs/ensembl.
		Up 0/0 DEGs/ensembl.
		Dw 0/0 DEGs/ensembl.

Found 0 (best=3) pathways for geneset num=0 'Reactome_Pathways_2024'
Pathway cutoffs p-value=0.050 fdr=0.050 min genes=0.05No enrichment analysis was calculated.


In [5]:
cbio = cBioPortal(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

### Get all programs

In [6]:
verbose = False

df_psi = cbio.open_primary_site(verbose=verbose)
df_psi

,prog_id,cbioportal_study_id,active,gdc_project_id,psi_id,disease_id,disease_cd,primary_site,disease_context
0,TCGA,paad_tcga_pan_can_atlas_2018,True,TCGA-PAAD,PAAD,pancreatic_adenocarcinoma,PAAD,Pancreas,"TCGA pancreatic adenocarcinoma, PanCancer Atlas"
1,CPTAC3,paad_cptac_2021,True,CPTAC-3,PAAD,pancreatic_ductal_adenocarcinoma,PAAD,Pancreas,"CPTAC publication cohort, Cell 2021; 140 pancreatic cancers"
2,TCGA,skcm_tcga_pan_can_atlas_2018,True,TCGA-SKCM,SKCM,cutaneous_melanoma,SKCM,Skin,"TCGA skin cutaneous melanoma, PanCancer Atlas"
3,TCGA,brca_tcga_pan_can_atlas_2018,True,TCGA-BRCA,BRCA,breast_invasive_carcinoma,BRCA,Breast,"TCGA breast invasive carcinoma, PanCancer Atlas"
4,CPTAC2,brca_cptac_2020,True,CPTAC-2,BRCA,breast_cancer,BRCA,Breast,"CPTAC breast cancer publication cohort, Cell 2020"


### Open primary cites from cbio

In [7]:
PROG_ID = 'TCGA'
psi_id = 'PAAD'
psi_id = 'SKCM'
psi_id = 'BRCA'

PROG_ID = 'CPTAC2'
psi_id = 'BRCA'

PROG_ID = 'CPTAC3'
psi_id = 'PAAD'

### Prism - development

In [8]:
import anndata as ad

prism = PRISM(root0=ROOT0, root0_data=ROOT0_DATA)

verbose=True

prism.set_program_and_primary_site(prog_id=PROG_ID, psi_id=psi_id, verbose=verbose)

prism.root_singc, prism.root_singc.exists()

Table opened ((7, 9)) at '/home/flavio/uv/perturb_agent/data/cbioportal_study_mapping.tsv'

-----------------------------
>> prog_id: CPTAC3
>> psi_id: PAAD
>> primary_site: Pancreas
>> disease_id: pancreatic_ductal_adenocarcinoma
>> disease_cd: PAAD

-----------------------------
>> cbioportal_study_id: paad_cptac_2021
>> gdc_project_id: CPTAC-3

-----------------------------
>> root disease: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD
>> root samples: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/samples
>> root lfc: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/lfc
>> root mutations: /home/flavio/uv/perturb_agent/data/CPTAC3/PAAD/mutations
-----------------------------



(PosixPath('/home/flavio/uv/perturb_agent/data/single_cell'), True)

### Running prism

In [9]:
verbose=True

res = prism.open_bayesprism(verbose=verbose)
print(len(res.genes))

Loaded /home/flavio/uv/perturb_agent/data/single_cell/deconv.h5ad (6.8 MB)
1604


In [10]:
res.states

['Fibroblast cell',
 'Stellate cell',
 'Macrophage cell',
 'Endothelial cell',
 'T cell',
 'B cell',
 'Ductal cell type 2',
 'Endocrine cell',
 'Ductal cell type 1',
 'Acinar cell']

In [11]:
res.theta_type.columns

Index(['Acinar cell', 'B cell', 'Ductal cell type 1', 'Endocrine cell', 'Endothelial cell',
       'Fibroblast cell', 'Macrophage cell', 'Stellate cell', 'T cell', 'malignant'],
      dtype='object')

### Ductal cell type 1

"Ductal cell type 1" is the normal-like ductal population and stays in the environment compartment — which is what you want. If both had been mapped to malignant, purity would inflate. Verify with res.tumor_purity.groupby(meta["condition"]).describe(): normals near zero, tumors somewhere in 0.2–0.6.


### Ductal cell type 2

One malignant state means no Ductal cell type 2 subdivision, so subtype_malignant scores Moffitt signatures on a single pooled malignant profile. That still works — it's per-sample expression, so samples can differ — but it won't give you distinct malignant states in θ. For that you'd subcluster Ductal cell type 2 in the AnnData and write finer cell_state labels before calling pseudobulk_reference.

In [12]:
res.cell_type_expression("Ductal cell type 1").shape

(1604, 153)

In [13]:
res.cell_type_expression("Ductal cell type 2").shape

(1604, 153)

### 2. theta is now fixed -> expand Z to every gene

In [14]:
verbose=False
force=False

imax_tumor=250
imax_normal=50

exclude_prog_list=['CCLE']
disease_cd = 'PAAD'

dfn_tumor, dfn_normal, df_gtex, df_summ = cbio.get_all_data_from_disease(disease_cd=disease_cd, 
                                                           imax_tumor=imax_tumor, imax_normal=imax_normal,
                                                           exclude_prog_list=exclude_prog_list,
                                                           force=force, verbose=verbose)

verbose=True
force=False

df_bulk, df_meta = prism.build_bulk_matrix(dfn_tumor, dfn_normal, cbio.df_metadata, 
                                        keep_biotypes=("protein_coding", "lncRNA", "miRNA"),
                                        force=force, verbose=verbose)

force=False
verbose=True
fname = "count-matrix.txt"
fname_ad = fname.replace('.txt', '.h5ad')

adata = prism.load_matrix(fname=fname, sep=' ', force=force, verbose=verbose)

filename_ad = prism.root_singc / fname_ad
compression = "gzip"
# adata.write_h5ad(filename_ad, compression=compression)
print(f"AData saved as {filename_ad},  ({filename_ad.stat().st_size/1e6:.0f} MB), compressed with {compression}")


verbose=True
fname_celltype = "all_celltype.txt"
adata = prism.attach_celltypes(adata=adata, fname_celltype=fname_celltype, verbose=verbose)

ref, s2t = prism.pseudobulk_reference(adata)

Error reading csv/tsv '/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/lfc/expression_gtex_controls_counts.tsv': No columns to parse from file
Table opened ((27169, 153)) at '/home/flavio/uv/perturb_agent/data/single_cell/bulk_matrix.tsv'
Table opened ((153, 4)) at '/home/flavio/uv/perturb_agent/data/single_cell/bulk_metadata.tsv'
57,530 cells x 24,005 genes | obs: []
AData saved as /home/flavio/uv/perturb_agent/data/single_cell/count-matrix.h5ad,  (338 MB), compressed with gzip
all_celltype.txt columns: ['cluster']
                             cluster
cell.name                           
T1_AAACCTGAGATGTCGG  Fibroblast cell
T1_AAACGGGGTCATGCAT    Stellate cell
T1_AAAGATGCATGTTGAC  Macrophage cell
using type_col='cluster'
barcode overlap: 57,530 / 57,530
cell_type
malignant             11315
Ductal cell type 1    10317
Endothelial cell       9117
Fibroblast cell        6742
Stellate cell          5907
Macrophage cell        5361
T cell                 3660
B cell                 24

In [ ]:
Zfull, gfull = prism.full_Z(res, df_bulk, ref)

In [ ]:
dic = {}

for cell_state in res.states:
    print(cell_state)
    Z = prism.state_expression(Zfull, gfull, res, cell_state)
    dic[cell_state] = Z

In [ ]:
i=0
key = list(dic.keys())[i]

print(key)
dic[key]

### Ductal 2 - malignant

In [ ]:
Zmal = prism.state_expression(Zfull, gfull, res, "Ductal cell type 2")

In [ ]:
for g in ["FAM83A-AS1", "HOXA10-AS", "HOXB-AS3", "MIR7-3HG"]:
    if g in gfull:
        print(g, prism.gene_compartment_share(Zfull, gfull, res, g).head(3).round(3).to_dict())

In [ ]:
prog1 = ["FAM83A-AS1", "HOXA10-AS", "HOXB-AS3", "HOXB-AS4", "MIR7-3HG"]

prog2 = ["GATA6", "KRT17", "NEAT1", "H19", "DLEU1", "DLEU2"]

### survived build_bulk_matrix?

> Almost certainly df_bulk is the culprit: build_bulk_matrix defaults to keep_biotypes=("protein_coding",), which removes every lncRNA. Rebuild with them included:

In [ ]:
[g for g in prog1 if g in df_bulk.index]

### present in the scRNA reference?

In [ ]:
  
[g for g in prog1 if g in ref.columns]

### Data treatment

1. get raw dfc
2. filter low-expression genes
3. normalize for library size
4. variance-stabilizing transformation
5. select most variable genes
6. cluster samples into k = 3..8 groups
7. evaluate clusters
8. find gene signatures for each cluster

A low-expression gene can be biologically important and even differentially expressed, especially if it is a transcription factor, cytokine, receptor, lncRNA, or rare-cell marker.

But for unsupervised tumor clustering, we usually do not want thousands of genes with mostly zero/very low counts because they add noise and unstable distances.

In [ ]:
set(ref.index.to_list())

In [ ]:
set(s2t.index.to_list())

In [ ]:
adata.obs

In [ ]:
import re, numpy as np, pandas as pd

adata.obs["sample"] = adata.obs_names.to_series().str.extract(r"^([TN]\d+)_")[0].values
adata.obs["tissue"] = np.where(adata.obs["sample"].str.startswith("T"), "tumor", "normal")

print(adata.obs.groupby("tissue")["sample"].nunique())      # expect tumor 24, normal 11
print(pd.crosstab(adata.obs["cell_state"], adata.obs["tissue"]))

### Count Malignant Cells - accordingo to transcriptomics

In [ ]:
d2 = adata.obs["cell_state"].eq("Ductal cell type 2")
print(len(d2))
d2[:5]

In [ ]:
is_t = adata.obs["tissue"].eq("tumor")
print(np.sum(is_t))

In [ ]:
adata.obs["cell_state"] = np.where(d2 &  is_t, "Malignant ductal",
                          np.where(d2 & ~is_t, "Ductal cell type 2 normal",
                                   adata.obs["cell_state"]))
adata.obs

In [ ]:
from collections import Counter

Counter(adata.obs["cell_state"] )

In [ ]:
adata.obs["cell_type"]  = np.where(adata.obs["cell_state"].eq("Malignant ductal"),
                                   "malignant", adata.obs["cell_type"])

Counter(adata.obs["cell_type"] )

In [ ]:
ref2, s2t = prism.pseudobulk_reference(adata, state_key="cell_state", type_key="cell_type")
s2t.to_dict()

### LFC

calc_celltype_lfc() — each compartment vs the mean of the others, paired across samples by default. Paired is the right default here because every sample contributes every cell type, so pairing removes cohort/purity variance. This doubles as deconvolution QC: if the ductal compartment doesn't recover KRT19/TFF1/CEACAM6 and fibroblast doesn't recover COL1A1/POSTN, θ or the Peng reference is off and step 2 is meaningless.

### Critics

- Why not, for each cell type, tumor samples x normal samples
- Only Ductal 2 Tumor has no normal samples - to confirm


In [ ]:
res.__dict__.keys()

In [ ]:
res.states

### Prism programs

In [ ]:
Z_full, genes_full = prism.full_Z(res, df_bulk, ref)

### MalignantCluster

In [ ]:
# del(MalignantCluster)

In [ ]:
from libs.prism_malig_lib import MalignantCluster

In [ ]:
type(res)

In [ ]:
cbio.root_mprog_disease

In [ ]:
root_mprog_cluster = create_dir(cbio.root_mprog_disease, 'cluster')

cell_name = "Ductal cell type 2"
kmax = 8
no_decouple = True
is_tahoe = True
'''
mc = MalignantCluster(prism=prism, res=res, df_bulk=df_bulk, ref=ref, 
                      root_mprog_cluster=root_mprog_cluster, 
                      organ="Pancreas", cell_name=cell_name, cell_types=None)
'''

import importlib, libs.prism_malig_lib as pml
importlib.reload(pml)
print(pml.__version__)

In [ ]:
mc = pml.MalignantCluster(prism, res, df_bulk, ref, root_mprog_cluster, organ="Pancreas")

X, diag = mc.prepare_malignant_matrix(decouple_purity=False, keep_genes=mc.program1_panel, drop_pattern=r"^N-")
print(X.shape)
X.head(3)

In [ ]:
lista = [x for x in X.index if x.startswith('T-')]
X.shape[0], len(lista) == X.shape[0]

In [ ]:
diag.keys()

In [ ]:
diag["samples_excluded_by_filter"]

In [ ]:
diag["pc_theta_pearson_raw"]   # PC-vs-theta on logx (pre-decoupling)

In [ ]:
diag["pc_theta_note"]          # warns the decoupled version is ~0 by construction

In [ ]:
diag["sample_mean_expr"]       # Xc.mean(axis=1) per sample

In [ ]:
diag["sample_total_Z"]         # ms.Z.sum(axis=1) per sample

In [ ]:
diag["theta_excluded"]

In [ ]:
diag["theta_kept"]

### Inspecting vars

In [ ]:
import inspect
print(pml.__version__)
print("drop_pattern" in inspect.signature(mc.prepare_malignant_matrix).parameters)

In [ ]:
info = mc.inspect_de_schema(genes=X.columns)
print(info.keys())
info["columns"]

In [ ]:
info["matches"]

In [ ]:
info["dtypes"]

In [ ]:
info["resolved_columns"]

In [ ]:
info["numeric_profile"]      # min / max / mean / frac_negative / n_unique


In [ ]:
info["signed_candidates"]

In [ ]:
cov = mc.index_coverage()
cov["n_lines_seen"], cov["n_lines_in_metadata"]

In [ ]:
cov["block_probe_counts"]        # min probes per block

In [ ]:
lista = cov["in_metadata_not_in_index"]
len(lista), lista[:5]

In [ ]:
cov["n_lines_seen"], cov["n_lines_in_metadata"]

### Shards - A database shard, or simply a shard, is a horizontal partition of data within a database or search engine.

In [ ]:
cvcl = mc.organ_cell_lines() 
mc.shard_index(stride=6)          # ~120 new footers, ~9 min (not 172)
shards = mc.find_shards_for(cvcl) # expect: located 11, missing 0

In [ ]:
len(cvcl), cvcl[:3]

In [ ]:
dropped = ('CVCL_0186','CVCL_1634','CVCL_1638','CVCL_1639')
cvcl7  = [c for c in cvcl if c not in dropped]
shards = mc.find_shards_for(cvcl7)          # expect missing: 0

In [ ]:
mc.shard_sizes(shards)

In [ ]:
mc.shard_sizes(shards)["bytes"].sum() / 1e9      # GB for the 237 shards

In [ ]:
# to much: lets paralelize
# mc.download_shards(shards, dry_run=True)
mc.download_shards(shards, max_workers=8)

In [ ]:
"""
df["condition"] = df["cell_line_id"].astype(str) + "|" + df["drug"].astype(str)
df_pivot = (df.pivot(index="gene", columns="condition", values="stat")
        .astype(dtype))
"""

df_pivot, cond = mc.load_tahoe_de(genes=X.columns.to_list(), organs=("Pancreas",),
                                  mode="download", _shard_subset=shards, force=True)

In [ ]:
print(df_pivot.shape)                                  # genes x conditions
df_pivot.head(8)

In [ ]:
cond.head(3)

In [ ]:
cond["cell_line_id"].nunique(), cond["drug"].nunique()

In [ ]:
cond["cell_line_id"].value_counts()     # expect 7 lines

In [ ]:
cond["drug"].value_counts()


In [ ]:
print("\n".join(np.unique(cond["drug"])))

In [ ]:
## 2D-values, stacked distribution
df_pivot.stack().describe()

In [ ]:
np.sum(np.sum(df_pivot<=1))

In [ ]:
np.sum(np.sum(df_pivot<=-1))

In [ ]:
float((df_pivot <= -1).mean().mean())

In [ ]:
(df_pivot >= 1).mean(axis=0)

In [ ]:
(df_pivot >= 1).mean(axis=1)

### Cluster

#### All three checks pass

- 0.4989 negative confirms stat is a genuinely signed statistic 
- the WTCS sign convention is sound. 
- 1986 of 2000 HVGs found in Tahoe is 99.3% coverage, better than I expected for a Parse 3' assay.

#### Score it:

In [ ]:
cc = mc.consensus_cluster(X)
cc

### PAC

Proportion of Ambiguous Clustering — a measure of how decisively the consensus clustering assigns samples, from Șenbabaoğlu et al. (2014). 

It's how choose_k picks k.

The consensus matrix C[i,j] is the fraction of resamples in which samples i and j landed in the same cluster, given both were drawn. 

Perfect structure gives values of 0 or 1 — pairs always together or always apart. Unstable structure gives values scattered in between.

PAC is just the fraction of pairs sitting in that ambiguous middle:



In [ ]:
consensus = cc[2]['consensus']
consensus.iloc[:5, :10]


In [ ]:
def _pac(consensus, lo=0.1, hi=0.9):
    v = consensus[np.triu_indices_from(consensus, k=1)]
    return float(((v > lo) & (v < hi)).mean())

_pac(consensus.values, lo=0.1, hi=0.9)

Low PAC = crisp, reproducible partition. choose_k takes the smallest k within pac_tol of the minimum, preferring parsimony when several k are comparably stable.

Why it misled you here. PAC rewards reproducibility, not biological meaning, and those come apart badly for unbalanced splits. Peeling six outliers off 119 samples is maximally reproducible — every resample isolates them identically — so PAC approaches zero. The metric is behaving exactly as designed while pointing at nothing interesting.

It's also mechanically biased toward small k, since fewer clusters means fewer boundaries to disagree about. That's why choose_k at k=2 deserves scepticism rather than confidence on its own.

So read the diagnostics together:

In [ ]:
dfa = pd.DataFrame({k: {"pac": v["pac"], "coph": v["cophenetic"],
                    "sil": v["silhouette"],
                    "sizes": v["labels"].value_counts().tolist()}
                    for k, v in cc.items()}).T

dfa

The sizes column is the one that would have caught this. 

A k with low PAC and balanced clusters is trustworthy; 
low PAC with a 6/119 split is an outlier detector. 

Cophenetic correlation and silhouette are worth glancing at too, though both share the same blind spot — none of them knows the difference between a real subtype and six weird samples.

In [ ]:
k = mc.choose_k(cc)
k

### k=2 is the expected answer for PDAC 

- Moffitt's classical vs basal-like is a two-group axis. 
- The earlier problem wasn't k=2, it was the 6-vs-119 split. 
- So the question now is what the two groups are.

Three checks, in order of how much they'd change your interpretation:

In [ ]:
labels = cc[2]["labels"]
print('n counts', labels.value_counts().to_dict())          # balanced now?
print("")
print(diag["pc_theta_pearson"])                 # PC1 vs theta_mal
print("")
print(pd.crosstab(labels, pd.Series(
    ['TCGA' if 'TCGA' in s else 'CPTAC' for s in X.index], index=X.index)))

A near-even split with |r| below ~0.3 on PC1 is what you want.

- If the crosstab shows the split tracking TCGA vs CPTAC, it's a batch axis — plausible given your strandedness history, 
- and it would mean the unstranded harmonisation didn't fully remove the cohort effect.

Then the test that actually names the clusters:

In [ ]:
basal = ["KRT81","KRT5","KRT6A","KRT17","S100A2","SPRR3","TP63","DHRS9","VGLL1"]
clas  = ["GATA6","TFF1","TFF2","TFF3","LGALS4","CLDN18","CEACAM6","AGR2", "ANXA10","REG4","CTSE","MUC13"]
Z = (X - X.mean()) / X.std()
sc = pd.DataFrame({
    "basal":     Z[[g for g in basal if g in X.columns]].mean(axis=1),
    "classical": Z[[g for g in clas  if g in X.columns]].mean(axis=1)})
print(sc.groupby(labels).mean().round(2))

If **one cluster is basal-high/classical-low** and **the other the reverse**, you've recovered Moffitt in the deconvolved malignant compartment

- which is a genuinely stronger result than the bulk clustering you started with, 
- because it's not confounded by stromal content. That was the whole point of the deconvolution detour.

If instead both clusters co-elevate the two programs, 
- you're seeing the same cellularity axis as before, and the purity decoupling didn't clear it.

Note how few of those markers likely survived your HVG filter — check [g for g in basal+clas if g in X.columns] first. If coverage is thin, score on the un-HVG-filtered logx instead, since marker scoring doesn't need the variance selection.

In [ ]:
labels = cc[k]["labels"]
sig    = mc.cluster_signatures(X, labels)
sig[1]

In [ ]:
"; ".join(sig[1]['stat'].index.to_list())

In [ ]:
"; ".join(sig[2]['stat'].index.to_list())

### Critic

In your notebook, cell 72's output showed 'n': 6 and 'n': 119 - the n field cluster_signatures records for each cluster. 

So choose_k picked k=2, but the two groups were 6 samples and 119 samples, not two comparable halves.

In [ ]:
sig[1]['n'], sig[2]['n']

### Each signature

In [ ]:

clu=2
sig[clu].keys()

In [ ]:
sig[clu]['n'], len(sig[clu]['up']), len(sig[clu]['down'])

In [ ]:
ups = set(sig[clu]['up'])
dwns = set(sig[clu]['down'])

ups.intersection(dwns), len(ups.union(dwns))

In [ ]:
clu=1
sig[clu].keys()

In [ ]:
sig[clu]['n'], len(sig[clu]['up']), len(sig[clu]['down'])

In [ ]:
ups = set(sig[clu]['up'])
dwns = set(sig[clu]['down'])

ups.intersection(dwns), len(ups.union(dwns))

In [ ]:
sig[clu]['up']

### signature_table()

In [ ]:
len(labels), labels

In [ ]:
dft = mc.signature_table(X, labels=labels,  min_abs_lfc=0.6, sort_by='fdr_nominal')
dft['abs_lfc'] = dft['lfc'].abs()

fdr_cutoff = 0.05
lfc_cutoff = 1

dft = dft[(dft.fdr_nominal < fdr_cutoff) & (dft.abs_lfc >= lfc_cutoff)]
dft = dft.sort_values('fdr_nominal', ascending=True)

dft.shape

In [ ]:
dft.fdr_nominal.hist()

In [ ]:
dft.head(6)

In [ ]:
genes_sel = sig[clu]['up']

df1 = dft[dft.gene.isin(genes_sel)]
print(f"Number of upregulated genes in cluster {clu}: {df1.shape[0]}")
df1

In [ ]:
genes_clu = np.unique(df1.gene)
print(len(genes_clu))


In [ ]:
genes_clu_in = [x for x in genes_clu if x in df_pivot.index]
df2 = df_pivot.loc[genes_clu_in]

print(df2.shape)
df2.T

### WTCS (Weighted Connectivity Score) and NCS (Normalized Connectivity Score)

WTCS (Weighted Connectivity Score) and NCS (Normalized Connectivity Score) are core metrics used in the LINCS and Connectivity Map (CMap) pipelines to compare query gene signatures against reference expression profiles. 

- WTCS measures signature similarity from −1 to 1
- NCS normalizes these scores within specific cell lines and perturbagen types.

In [ ]:
cond

In [ ]:
cond.index.is_unique 

In [ ]:
cond = cond[~cond.index.duplicated()].loc[df_pivot.columns]
print(cond.index.is_unique)
cond.head(3)

In [ ]:
R1 = mc.score_clusters_vs_tahoe(sig, df_pivot, cond)
R1

In [ ]:
len(R1.targets.unique()), R1.targets.unique()[:20]

In [ ]:
target_list = R1.targets.unique()
len(target_list), len(genes_clu)

In [ ]:
[x for x in target_list if x in genes_clu]

In [ ]:
R1a = R1[ (R1.cluster == clu) & (R1.targets.isin(genes_clu)) & (R1.wtcs.abs() > 0.1) ]
print(R1a.shape)
R1a

In [ ]:
R1a.targets.unique()

In [ ]:
R1a.groupby("cluster").head(10)[["cluster","cell_name","drug","moa-fine","ncs"]]   # reversers

In [ ]:
R1a.groupby("cluster").tail(10)[["cluster","cell_name","drug","moa-fine","ncs"]] 

In [ ]:
out = mc.run(ks=range(2, 9), drop_pattern=r"^N-", min_share=0.3)     # tune from diagnose_filters()
R   = mc.score_clusters_vs_tahoe(out["signatures"], df_pivot, cond)

In [ ]:
R.groupby("cluster").head(10)[["cluster","cell_name","drug","moa-fine","ncs"]]   # reversers

In [ ]:
R.groupby("cluster").tail(10)[["cluster","cell_name","drug","moa-fine","ncs"]] 

In [ ]:
# mc.root_mprog_tahoe = create_dir(mc.root_mprog_cluster / "tahoe")
mc.root_mprog_tahoe

In [ ]:
d = mc.root_mprog_tahoe / "metadata" / "pseudobulk_differential_expression"
files = sorted(d.glob("*.parquet"))
len(files), sum(f.stat().st_size for f in files) / 1e9

In [ ]:
cl = pd.read_parquet(mc.root_mprog_tahoe / "metadata"/ "cell_line_metadata.parquet")
cl[cl.Organ=="Pancreas"][["Cell_ID_Cellosaur","cell_name"]]

In [ ]:
d = mc.diagnose_filters()

d["Z_looks_like_counts"], d["Z_median_of_medians"]


In [ ]:
d["by_min_counts"]

In [ ]:
d["by_min_share"]

In [ ]:
d["joint_grid"]